# Task 7 — Apply Homographic Reprojection to YOLO26n Detections

Apply Joao's homography matrices to YOLO26n detections,
converting pixel coordinates to real-world lat/lon for each dataset.

In [2]:
import os
os.chdir('..')

In [3]:
import sys
!{sys.executable} -m pip install folium pyproj -q


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [4]:
import numpy as np
import cv2
import pandas as pd
import folium
from folium.plugins import TimestampedGeoJson
from pyproj import Transformer
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

## Step 1 — Ground Control Points (GCPs) per dataset

These GCPs were defined by Joao in his reprojection scripts.

In [5]:
SHAPEFILE_EPSG = 26916
LOCAL_TZ = ZoneInfo("America/Chicago")

DATASET_GCPS = {
    "W042_08-09-2025": {
        "image_points": np.float32([
            [910,  708],   # GCP 1
            [1502, 708],   # GCP 2
            [1921, 1724],  # GCP 3
            [296,  1853],  # GCP 4
        ]),
        "world_points": np.float32([
            [445447.469, 4617438.348],  # GCP 1
            [445442.892, 4617423.431],  # GCP 2
            [445408.560, 4617435.365],  # GCP 3
            [445409.454, 4617442.861],  # GCP 4
        ]),
    },
    "W065_08-17-2025": {
        "image_points": np.float32([
            [1396, 935],   # GCP 1
            [1786, 899],   # GCP 2
            [2060, 1336],  # GCP 3
            [933,  1216],  # GCP 4
            [527,  1429],  # GCP 5
            [326,  1194],  # GCP 6
        ]),
        "world_points": np.float32([
            [436606.877, 4649769.770],  # GCP 1
            [436617.370, 4649763.091],  # GCP 2
            [436591.414, 4649748.604],  # GCP 3
            [436586.549, 4649763.644],  # GCP 4
            [436579.372, 4649762.214],  # GCP 5
            [436579.870, 4649774.438],  # GCP 6
        ]),
    },
    "W06E_12-02-2025": {
        "image_points": np.float32([
            [1220, 890],   # GCP 1
            [819,  765],   # GCP 2
            [877,  843],   # GCP 3
            [2130, 1185],  # GCP 4
            [659,  1185],  # GCP 5
        ]),
        "world_points": np.float32([
            [434756.062, 4648444.831],  # GCP 1
            [434756.471, 4648495.357],  # GCP 2
            [434749.445, 4648456.210],  # GCP 3
            [434756.651, 4648418.259],  # GCP 4
            [434738.281, 4648429.248],  # GCP 5
        ]),
    },
}

# Compute homography matrix for each dataset
homographies = {}
for dataset, gcps in DATASET_GCPS.items():
    H, _ = cv2.findHomography(gcps["image_points"], gcps["world_points"])
    homographies[dataset] = H
    print(f"✓ {dataset} — homography computed")

✓ W042_08-09-2025 — homography computed
✓ W065_08-17-2025 — homography computed
✓ W06E_12-02-2025 — homography computed


## Step 2 — Load YOLO26n detections

In [ ]:
yolo26n_df = pd.read_csv("/Users/best/cities-in-motion-video/keira/yolo26n_detections.csv")
print(f"Loaded {len(yolo26n_df)} YOLO26n detections")
print(f"Datasets: {yolo26n_df['dataset'].unique()}")
yolo26n_df.head()

## Step 3 — Apply homography to compute centroid and lat/lon

In [7]:
transformer = Transformer.from_crs(f"epsg:{SHAPEFILE_EPSG}", "epsg:4326", always_xy=True)

def reproject_centroid(pixel_point, H):
    pt = np.float32([[pixel_point]])
    return cv2.perspectiveTransform(pt, H)[0][0]

reprojected_rows = []

for dataset, group in yolo26n_df.groupby("dataset"):
    H = homographies.get(dataset)
    if H is None:
        print(f"[skip] No homography for {dataset}")
        continue

    for _, row in group.iterrows():
        cx = (row["x_min"] + row["x_max"]) / 2
        cy = (row["y_min"] + row["y_max"]) / 2

        world = reproject_centroid([cx, cy], H)
        lon, lat = transformer.transform(float(world[0]), float(world[1]))

        # Parse timestamp from timestamps_ns
        try:
            ts = datetime.fromtimestamp(
                int(row["timestamps_ns"]) / 1_000_000_000, tz=timezone.utc
            ).astimezone(LOCAL_TZ)
        except Exception:
            ts = None

        reprojected_rows.append({
            "timestamp" : ts,
            "dataset"   : dataset,
            "model"     : row["model"],
            "class"     : row["class"],
            "confidence": row["confidence"],
            "pixel_cx"  : cx,
            "pixel_cy"  : cy,
            "lat"       : lat,
            "lon"       : lon,
        })

proj_df = pd.DataFrame(reprojected_rows)
print(f"\n✓ Reprojected {len(proj_df)} detections")
proj_df.head()


✓ Reprojected 3016 detections


,timestamp,dataset,model,class,confidence,pixel_cx,pixel_cy,lat,lon
0,2025-08-08 19:00:14.432150-05:00,W042_08-09-2025,yolo26n,car,0.842330,427.084473,858.372894,41.706941,-87.655899
1,2025-08-08 19:00:14.432150-05:00,W042_08-09-2025,yolo26n,car,0.796858,1077.839355,712.215424,41.706829,-87.655727
2,2025-08-08 19:00:14.432150-05:00,W042_08-09-2025,yolo26n,car,0.612094,1542.115173,848.701874,41.706783,-87.655946
3,2025-08-08 19:00:14.432150-05:00,W042_08-09-2025,yolo26n,car,0.495358,1166.985229,631.449188,41.706768,-87.655459
4,2025-08-08 19:00:14.432150-05:00,W042_08-09-2025,yolo26n,car,0.374008,955.954620,629.897064,41.706840,-87.655417


## Step 4 — Save reprojected data

In [ ]:
out_path = Path("/Users/best/cities-in-motion-video/keira/yolo26n_reprojected.csv")
proj_df.to_csv(out_path, index=False)
print(f"✓ Saved to {out_path}")

## Step 5 — Visualize on interactive map (per dataset)

In [ ]:
CLASS_COLORS = {
    "car"        : "#3498db",
    "truck"      : "#e74c3c",
    "bus"        : "#e67e22",
    "person"     : "#2ecc71",
    "pedestrian" : "#2ecc71",
    "bicycle"    : "#9b59b6",
    "motorcycle" : "#1abc9c",
    "traffic light": "#f1c40f",
    "train"      : "#8e44ad",
}

legend_html = """
<div style="
    position: fixed; bottom: 80px; left: 30px; z-index: 9999;
    background: rgba(0,0,0,0.85); color: white;
    font-family: 'Courier New', monospace; font-size: 13px;
    padding: 14px 18px; border-radius: 8px;
    border: 1px solid #555; line-height: 2;
">
    <b style="color:#ffff00; font-size:14px;">YOLO Detections</b><br>
    <span style="color:#3498db; font-size:18px;">●</span>&nbsp; Car<br>
    <span style="color:#e74c3c; font-size:18px;">●</span>&nbsp; Truck<br>
    <span style="color:#e67e22; font-size:18px;">●</span>&nbsp; Bus<br>
    <span style="color:#2ecc71; font-size:18px;">●</span>&nbsp; Person / Pedestrian<br>
    <span style="color:#9b59b6; font-size:18px;">●</span>&nbsp; Bicycle<br>
    <span style="color:#1abc9c; font-size:18px;">●</span>&nbsp; Motorcycle<br>
    <span style="color:#ffffff; font-size:18px;">●</span>&nbsp; Other<br>
    <hr style="border-color:#555; margin: 6px 0;">
    <span style="color:#aaaaaa; font-size:11px;">Click any point for details</span>
</div>
"""

for dataset, group in proj_df.groupby("dataset"):
    group = group.dropna(subset=["timestamp"])
    group = group.sort_values("timestamp")

    center = [group["lat"].mean(), group["lon"].mean()]

    m = folium.Map(
        location=center,
        zoom_start=19,
        max_zoom=21,
        tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
        attr="Google"
    )

    features = []
    for _, row in group.iterrows():
        features.append({
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [row["lon"], row["lat"]],
            },
            "properties": {
                "time"  : row["timestamp"].isoformat(),
                "popup" : (
                    f"<b>{row['class']}</b><br>"
                    f"Conf: {row['confidence']:.2f}<br>"
                    f"Time: {row['timestamp'].strftime('%H:%M:%S')}"
                ),
                "icon"  : "circle",
                "iconstyle": {
                    "fillColor"  : CLASS_COLORS.get(row["class"], "#ffffff"),
                    "fillOpacity": 1.0,
                    "stroke"     : True,
                    "color"      : "#000000",
                    "opacity"    : 1.0,
                    "weight"     : 2,
                    "radius"     : 6,
                },
            },
        })

    TimestampedGeoJson(
        {"type": "FeatureCollection", "features": features},
        period="PT5M",
        duration="PT5M",
        auto_play=True,
        loop=True,
        max_speed=5,
        loop_button=True,
        date_options="YYYY-MM-DD HH:mm:ss",
        time_slider_drag_update=True,
        add_last_point=False,
    ).add_to(m)

    m.get_root().html.add_child(folium.Element(legend_html))

    out_map = Path(f"/Users/best/cities-in-motion-video/keira/{dataset}_yolo26n_animated_map.html")
    m.save(str(out_map))
    print(f"✓ Map saved: {out_map}")